# P2 variant 2: two-step per-clause extraction, then synthesis

Variant 1 (`p2_structured_prompting.ipynb`) told a single generation call to "state
every condition, don't stop after the first" - it worked mechanically but didn't move
`llm_grade` correctness, and caused a greedy-decoding repetition loop on one row.

Variant 2 splits the task instead of instructing harder on the same task:
- **Step A (extraction)**: one short, constrained call per retrieved clause - "list
  every condition, requirement, or element THIS clause states relevant to the
  question, don't omit any."
- **Step B (synthesis)**: one call per query, combining the already-complete
  per-clause extractions into a final cited answer - no "don't stop" instruction
  needed, since completeness is now Step A's job.

Same 21 universal-failure queries, same `hybrid` retrieved clause IDs, same local 8B
generator, same RAGAS scoring pipeline as variant 1 - only the generation strategy
differs, isolating that as the variable.

Run from `p2_two_step_extraction.py` (three phases: `extract`, `synthesize`,
`score` - long-running local-GPU phases were launched as background processes, not
executed live in this notebook; see `plan/STATUS.md`'s 2026-07-10 entry for the full
run log). This notebook mirrors that script for reproducibility and adds the
baseline/variant-1/variant-2 comparison.

## Step 1: imports and paths

In [ ]:
import json
from pathlib import Path

RESULTS_DIR = Path("../results")

def load_jsonl(path):
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]


## Step 2: extraction phase (Step A)

210 short, constrained calls (21 queries x 10 retrieved clauses each): "list every
condition, requirement, or element this clause states relevant to the question, don't
omit any." Sequential single-item calls - an earlier attempt to batch these
(`batch_size=8` via the raw HF pipeline) was killed after 9+ minutes of zero output,
because this machine's local GPU is a tight 6GB slice already ~95% full with just the
model loaded, and batching thrashed against that VRAM pressure rather than genuinely
parallelizing. Reverted to the sequential pattern already proven reliable in variant 1.

Run via: `uv run python experiment/p2_two_step_extraction.py extract`

In [ ]:
extractions = load_jsonl(RESULTS_DIR / "p2v2_extractions.jsonl")
print(f"{len(extractions)} queries extracted")
n_relevant = sum(
    sum(1 for e in row["extractions"] if e.strip().upper() != "NOT RELEVANT")
    for row in extractions
)
n_total = sum(len(row["extractions"]) for row in extractions)
print(f"{n_relevant}/{n_total} retrieved clauses judged relevant on average across all queries")


## Step 3: synthesis phase (Step B)

21 calls, each combining one query's complete per-clause extractions into a final
cited JSON answer (same `answer`/`citations` schema as the rest of this project).
No repetition loops were observed here (confirmed by reading every answer and
checking length outliers) - variant 1's specific failure mode is gone.

Run via: `uv run python experiment/p2_two_step_extraction.py synthesize`

In [ ]:
answers = load_jsonl(RESULTS_DIR / "p2v2_two_step_answers.jsonl")
print(f"{len(answers)} answers synthesized, 0 generation errors")
lens = [len(a["answer"]) for a in answers]
print(f"answer length: min={min(lens)} max={max(lens)} avg={sum(lens)/len(lens):.0f}")


## Step 4: scoring (RAGAS faithfulness/answer_relevancy + correctness)

Same judge (`gpt-5.5`) and pipeline as variant 1 and the original baseline - see
`p2_structured_prompting.ipynb` for the full judge/embeddings setup detail (the
vertexai stub and `model_args` patch this project's `ragas==0.4.3` install needs).

Run via: `uv run python experiment/p2_two_step_extraction.py score`

In [ ]:
v2_scores = load_jsonl(RESULTS_DIR / "p2v2_two_step_scores.jsonl")
print(f"{len(v2_scores)} score rows (21 queries x 2 reference models)")


## Step 5: compare against baseline and variant 1

The actual result: does splitting extraction from synthesis do better than a single
call told to "state every condition"?

In [ ]:
fails = load_jsonl(RESULTS_DIR / "p1_universal_failures.jsonl")
queries = set(r["query"] for r in fails)

base_corr = load_jsonl(RESULTS_DIR / "correctness_scores.jsonl")
base_ragas = load_jsonl(RESULTS_DIR / "ragas_answer_scores.jsonl")
base_hybrid_corr = [r for r in base_corr if r["query"] in queries and r["config"] == "hybrid"]
base_hybrid_ragas = [r for r in base_ragas if r["query"] in queries and r["config"] == "hybrid"]

v1_scores = load_jsonl(RESULTS_DIR / "p2_structured_prompting_scores.jsonl")

def summarize(name, corr_rows, faith_rel_rows=None, is_variant=False):
    print(f"--- {name} ---")
    for ref in ["14b", "72b"]:
        if is_variant:
            rows = [r for r in corr_rows if r["reference_model"] == ref]
            grades = [r["llm_grade"] for r in rows if r["llm_grade"] is not None]
            faiths = [r["faithfulness"] for r in rows if r["faithfulness"] is not None]
            rels = [r["answer_relevancy"] for r in rows if r["answer_relevancy"] is not None]
        else:
            rows = [r for r in corr_rows if r["ref_model"] == ref]
            grades = [r["llm_grade"] for r in rows if r["llm_grade"] is not None]
            faiths = [r["faithfulness"] for r in faith_rel_rows if r["faithfulness"] is not None]
            rels = [r["answer_relevancy"] for r in faith_rel_rows if r["answer_relevancy"] is not None]
        n_correct = sum(1 for g in grades if g == 1.0)
        print(f"  vs {ref}: llm_grade avg={sum(grades)/len(grades):.3f} ({n_correct}/{len(grades)} correct), "
              f"faithfulness avg={sum(faiths)/len(faiths):.3f}, answer_relevancy avg={sum(rels)/len(rels):.3f}")

summarize("BASELINE (hybrid, original prompt)", base_hybrid_corr, base_hybrid_ragas, is_variant=False)
summarize("VARIANT 1 (structured/completeness prompt)", v1_scores, is_variant=True)
summarize("VARIANT 2 (two-step extraction + synthesis)", v2_scores, is_variant=True)


## Findings

**Result: worse than both baseline and variant 1, on every metric except correctness
(tied at the floor).** `llm_grade` correctness: 0/21 against both reference models
(baseline: 0/21 both; variant 1: 0/21 / 1/21). Faithfulness: 0.502 (baseline 0.561,
variant 1 0.570 - variant 2 is the worst of the three). Answer relevancy: 0.707
(baseline 0.798, variant 1 0.738 - again worst). Mean answer length also dropped below
baseline: 407 chars vs baseline's 453 (variant 1 averaged 649, inflated by its
repetition-loop outlier).

**Mechanism 1 - extraction-step paraphrase drift breaks faithfulness against the
literal source text.** RAGAS faithfulness checks the *synthesized answer's* claims
against the raw retrieved clause text, not the intermediate extractions. Even when an
extraction is a locally reasonable summary (verified by direct reading - extractions
were generally accurate, not hallucinated), rephrasing during extraction introduces
wording drift the faithfulness judge doesn't map cleanly back to the literal clause
text. Concrete example: row #1 (POCA s327/s333B/s338 authorised-disclosure question)
scored faithfulness 0.00 despite its extraction for the cited clause reading as an
accurate paraphrase of the source.

**Mechanism 2 - synthesis sometimes over-compresses despite having complete
material.** The opposite failure from variant 1 (which over-produced, to the point of
looping). Row #20 (MLR 2017 reg 34 correspondent-relationship question) had 9/10
extractions judged relevant by Step A, yet the final synthesized answer was a
72-character noun-phrase fragment - not a real sentence, far short of what the
available extracted material actually supported.

**Read: decomposition trades one problem for two smaller, different ones - not a net
win.** Both P2 variants are now exhausted. Neither the instruction-based nor the
decomposition-based approach to fixing truncation reliably moves the correctness
judge, and each introduces its own new failure mode. This sharpens the diagnosis
further: the bottleneck looks less like a fixable prompt/pipeline defect and more like
a real 8B capability ceiling against this judge's bar. P3 (cross-encoder reranking - a
genuinely different mechanism, not generation-side) or revisiting generator scale are
the more promising remaining directions.

Full write-up: `plan/report.md`'s 2026-07-10 entry, `plan/improvement_plan.md`'s P2
section.